In [1]:
# ============================================================
# ERA5 Pressure Levels (0.25°) — parallel companion download
# CHANGE 1 APPLIED: batched by HALF-YEAR instead of by MONTH
# Same dataset (reanalysis-era5-pressure-levels), same
# resume/validity logic. A full year of hourly data across 3
# pressure levels for all of India hits CDS's "request too
# large" limit, so this batches by half-year (Jan-Jun, Jul-Dec)
# instead — still ~2 requests/year instead of 12.
# ============================================================

import os
import time
import warnings
import threading
from datetime import date
from concurrent.futures import ThreadPoolExecutor, as_completed

import cdsapi
import xarray as xr

warnings.filterwarnings("ignore", category=xr.SerializationWarning)

# ------------------------------------------------------------
# Setup
# ------------------------------------------------------------

DOWNLOAD_DIR = r"downloads_era5_pressure"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# India Coordinates (CDS format: [North, West, South, East])
AREA = [37.1, 68.12, 6.75, 97.42]

ALL_MONTHS = [f"{m:02d}" for m in range(1, 13)]
DAYS = [f"{d:02d}" for d in range(1, 32)]
TIMES = [f"{h:02d}:00" for h in range(24)]

# Parallel downloads (recommended to keep low for CDS API)
N_WORKERS = 2

# ------------------------------------------------------------
# Dynamic date range
# ------------------------------------------------------------
# Download everything from Jan 1980 until two months before today
# (latest months are skipped because ERA5 production may lag)

START_YEAR, START_MONTH = 1980, 1
LAG_MONTHS = 2

today = date.today()
end_y, end_m = today.year, today.month

for _ in range(LAG_MONTHS):
    end_m -= 1
    if end_m == 0:
        end_m = 12
        end_y -= 1

# ------------------------------------------------------------
# Build list of available months for every year
# ------------------------------------------------------------

year_plan = []

for y in range(START_YEAR, end_y + 1):

    if y == START_YEAR and y == end_y:
        months = [f"{m:02d}" for m in range(START_MONTH, end_m + 1)]

    elif y == START_YEAR:
        months = [f"{m:02d}" for m in range(START_MONTH, 13)]

    elif y == end_y:
        months = [f"{m:02d}" for m in range(1, end_m + 1)]

    else:
        months = ALL_MONTHS

    year_plan.append((str(y), months))

# ------------------------------------------------------------
# Split every year into Jan-Jun (H1) and Jul-Dec (H2)
# ------------------------------------------------------------

H1_MONTHS = {f"{m:02d}" for m in range(1, 7)}
H2_MONTHS = {f"{m:02d}" for m in range(7, 13)}

period_plan = []

for year, months in year_plan:

    h1 = [m for m in months if m in H1_MONTHS]
    h2 = [m for m in months if m in H2_MONTHS]

    if h1:
        period_plan.append((year, "H1", h1))

    if h2:
        period_plan.append((year, "H2", h2))

print(f"Today is        : {today}")
print(f"Year range      : {year_plan[0][0]}  →  {year_plan[-1][0]}")
print(f"Requests needed : {len(period_plan)} half-year requests "
      f"(was ~{sum(len(m) for _, m in year_plan)} monthly requests before)")
print(f"Parallel workers: {N_WORKERS}")
print(f"Output dir      : {DOWNLOAD_DIR}")
print("Setup done.\n")

# ------------------------------------------------------------
# Variable specification
# ------------------------------------------------------------

VARIABLES = [
    (
        "u_wind_pl",
        "reanalysis-era5-pressure-levels",
        {
            "product_type": ["reanalysis"],
            "variable": ["u_component_of_wind"],
            "pressure_level": ["500",'850'],
            "data_format": "netcdf",
            "download_format": "unarchived",
        },
    ),
]

# ============================================================
# Helper Functions
# ============================================================

def is_valid_nc(fpath):
    """
    Verify that a downloaded NetCDF file is usable.

    Opens the file using xarray and checks:
      - the file can be opened successfully
      - at least one data variable exists
      - the variable contains data

    Returns
    -------
    True  -> file is valid
    False -> corrupted, empty, or unreadable
    """
    try:
        with xr.open_dataset(fpath, engine="netcdf4") as d:

            dvs = list(d.data_vars)

            if not dvs:
                return False

            if d[dvs[0]].size == 0:
                return False

        return True

    except Exception:
        return False


def scan_existing(name, var_dir):
    """
    Scan the download directory before starting downloads.

    This function enables resume capability by:

      1. Removing leftover partial (.part) files.
      2. Checking every expected half-year file.
      3. Keeping valid files.
      4. Deleting corrupted files.
      5. Returning only the periods that still need downloading.

    Returns
    -------
    needs        : list of (year, half, months) still required
    n_valid      : number of valid files already present
    n_missing    : files that don't exist yet
    n_corrupt    : corrupted files removed
    n_partial    : stale partial downloads removed
    """

    needs = []

    n_valid = 0
    n_missing = 0
    n_corrupt = 0
    n_partial = 0

    # Remove interrupted download fragments
    for f in (os.listdir(var_dir) if os.path.isdir(var_dir) else []):

        if f.endswith(".part") or ".part." in f:

            try:
                os.remove(os.path.join(var_dir, f))
                n_partial += 1

            except OSError:
                pass

    # Check every required half-year file
    for year, half, months in period_plan:

        outfile = os.path.join(var_dir, f"{name}_{year}_{half}.nc")

        if not os.path.exists(outfile):

            needs.append((year, half, months))
            n_missing += 1

        elif is_valid_nc(outfile):

            n_valid += 1

        else:

            try:
                os.remove(outfile)
            except OSError:
                pass

            needs.append((year, half, months))
            n_corrupt += 1

    return needs, n_valid, n_missing, n_corrupt, n_partial


# Lock so print statements from multiple threads never overlap
_print_lock = threading.Lock()


def safe_print(msg):
    """
    Thread-safe printing.

    Multiple download threads may finish simultaneously.
    This lock ensures only one thread prints at a time so
    progress messages remain readable.
    """
    with _print_lock:
        print(msg, flush=True)


def download_one_period(name, dataset, extras, year, half, months, var_dir):
    """
    Download one half-year of ERA5 data.

    Steps
    -----
    1. Build the CDS API request.
    2. Download into a temporary '.part' file.
    3. Rename to the final filename only after success.
    4. Delete temporary files if anything fails.

    Returns
    -------
    (year, half, success, error_message)
    """

    final_path = os.path.join(var_dir, f"{name}_{year}_{half}.nc")
    part_path = f"{final_path}.part.{threading.get_ident()}"

    request = {
        **extras,
        "year": [year],
        "month": months,
        "day": DAYS,
        "time": TIMES,
        "area": AREA,
    }

    try:

        client = cdsapi.Client(
            quiet=True,
            wait_until_complete=True
        )

        client.retrieve(dataset, request, part_path)

        os.replace(part_path, final_path)

        return (year, half, True, None)

    except Exception as e:

        if os.path.exists(part_path):

            try:
                os.remove(part_path)
            except OSError:
                pass

        return (year, half, False, str(e))


def download_variable(name, dataset, extras):
    """
    Download an entire ERA5 variable.

    Workflow
    --------
    1. Scan existing files.
    2. Skip already-valid downloads.
    3. Launch parallel download workers.
    4. Monitor completion.
    5. Print a final summary.

    Returns
    -------
    Number of failed downloads.
    """

    var_dir = os.path.join(DOWNLOAD_DIR, name)
    os.makedirs(var_dir, exist_ok=True)

    print("=" * 60)
    print(f"{name}  ({dataset})")
    print("=" * 60)

    t0 = time.time()

    print("  Scanning existing files...", end=" ", flush=True)

    needs, n_valid, n_missing, n_corrupt, n_partial = scan_existing(
        name,
        var_dir,
    )

    parts = [f"{n_valid} valid"]

    if n_missing:
        parts.append(f"{n_missing} missing")

    if n_corrupt:
        parts.append(f"{n_corrupt} corrupted (deleted)")

    if n_partial:
        parts.append(f"{n_partial} partial (deleted)")

    print(", ".join(parts))

    if not needs:

        elapsed = (time.time() - t0) / 60.0
        print(f"  → nothing to do  ({elapsed:.1f} min)\n")

        return 0

    print(
        f"  Downloading {len(needs)} half-year period(s) "
        f"with {N_WORKERS} parallel workers..."
    )

    n_done = 0
    n_fail = 0

    completed = 0
    total = len(needs)

    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:

        futures = {
            pool.submit(
                download_one_period,
                name,
                dataset,
                extras,
                year,
                half,
                months,
                var_dir,
            ): (year, half)

            for year, half, months in needs
        }

        for fut in as_completed(futures):

            year, half, ok, err = fut.result()

            completed += 1

            label = f"{year}-{half}"

            if ok:

                n_done += 1

                safe_print(
                    f"    [{completed:>3}/{total}] {label}: done"
                )

            else:

                n_fail += 1

                err_short = (
                    err[:200] + "..."
                    if len(err) > 200
                    else err
                )

                safe_print(
                    f"    [{completed:>3}/{total}] "
                    f"{label}: FAILED: {err_short}"
                )

                safe_print(
                    f"      → if this is still a volume/field-limit "
                    f"error, split {label}"
                )

                safe_print(
                    f"        into individual months and retry."
                )

    elapsed = (time.time() - t0) / 60.0

    summary = (
        f"  → {name}: {n_done} downloaded, "
        f"{n_valid} pre-existing valid"
    )

    if n_corrupt:
        summary += f", {n_corrupt} corrupted re-attempted"

    if n_fail:
        summary += f", {n_fail} FAILED"

    summary += f"  ({elapsed:.1f} min)"

    print(summary + "\n")

    return n_fail


# ============================================================
# Main
# ============================================================

if __name__ == "__main__":

    overall_t0 = time.time()

    fail_summary = {}

    for name, dataset, extras in VARIABLES:

        fails = download_variable(name, dataset, extras)

        fail_summary[name] = fails

    total_min = (time.time() - overall_t0) / 60.0

    print("=" * 60)
    print("ERA5 Pressure Levels (India) — COMPLETE")
    print("=" * 60)
    print(f"Total wall time: {total_min:.1f} min")

    any_fails = sum(fail_summary.values())

    if any_fails:

        print(f"\n⚠ Failures by variable: {fail_summary}")
        print("  Re-run this cell — already-valid files will be skipped.")

    else:

        print("\n✓ No failures.")

Today is        : 2026-07-05
Year range      : 1980  →  2026
Requests needed : 93 half-year requests (was ~557 monthly requests before)
Parallel workers: 2
Output dir      : downloads_era5_pressure
Setup done.

u_wind_pl  (reanalysis-era5-pressure-levels)
  Scanning existing files... 91 valid, 2 missing


KeyboardInterrupt: 

5bde491b24b3fe8e34dcfcb8bbda37cf.nc:   0%|          | 0.00/214M [00:00<?, ?B/s]

e1ef5fb3a662504fc2d002b43b1ae4f8.nc:   0%|          | 0.00/266M [00:00<?, ?B/s]